In [ ]:
!pip install transformers accelerate bitsandbytes langchain sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 58.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 23.1 MB/s eta 0:0

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
print('Not connected to a GPU')
else:
print(gpu_info)


from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
print('Not using a high-RAM runtime')
else:
print('You are using a high-RAM runtime!')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from transformers import (
    MarkupLMFeatureExtractor,
    MarkupLMTokenizerFast,
    MarkupLMProcessor,
    MarkupLMForQuestionAnswering,
    AutoProcessor,
)
from tqdm.notebook import tqdm
from langchain import LLMChain, PromptTemplate
from langchain.memory import ConversationBufferWindowMemory, ConversationBufferMemory

import requests

In [ ]:
from bs4 import BeautifulSoup


def remove_nav_and_header_elements(content: BeautifulSoup) -> str:
    # Find all 'nav' and 'header' elements in the BeautifulSoup object
    nav_elements = content.find_all("nav")
    header_elements = content.find_all("header")

    # Remove each 'nav' and 'header' element from the BeautifulSoup object
    for element in nav_elements + header_elements:
        element.decompose()

    return str(content.get_text())

In [ ]:
# from transformers import MarkupLMFeatureExtractor
from bs4 import BeautifulSoup

# Initialize the feature extractor
feature_extractor = MarkupLMFeatureExtractor()

# Array to hold the entire corpus
multi_html_docs = []

# Set to keep track of processed URLs and avoid cycles
processed_urls = set()


def process_html(url):
    html_data = ""

    try:
        response = requests.get(url)
        response.raise_for_status()  # Ensure the request was successful
        html_data = response.text
        soup = BeautifulSoup(html_data, "html.parser")

    except requests.exceptions.MissingSchema:
        # skip if not a normal image file
        print("Invalid URL: ", url)
        return
        # prev = soup.select('a[rel="prev"]')[0]
        # print("prev link: ", prev)
    except requests.exceptions.HTTPError:
        print("Bad URL response: ", url)
        return
    # Append the HTML to the corpus array
    multi_html_docs.append(html_data)

    # Parse the HTML to find links to other documents
    links = [link["href"] for link in soup.find_all("a", href=True)]

    # Recursively process each linked document
    for link in links:
        # Avoid processing the same page multiple times
        if link not in processed_urls:
            processed_urls.add(link)
            process_html(link)


def process_sitemap(url):
    response = requests.get(url)
    response.raise_for_status()
    xml = response.text

    # Parse the XML to extract URLs
    # breakpoint()
    soup = BeautifulSoup(xml, "lxml")
    urls = [loc.text for loc in soup.find_all("loc")]

    # Process each URL
    for url in tqdm(urls, desc="sitemap links"):
        process_html(url)

In [ ]:
import nest_asyncio

nest_asyncio.apply()

from langchain.document_loaders.sitemap import SitemapLoader

sitemap = SitemapLoader(
    "https://docs.nvidia.com/nvidia-sitemap-202310.xml",
    # filter_urls=["https://api.python.langchain.com/en/latest/"],
    parsing_function=remove_nav_and_header_elements,
    continue_on_failure=True,
)


In [ ]:
docs = sitemap.load()

In [ ]:
len(docs)

In [ ]:
# Process each sitemap
sitemap_urls = [
    # "https://docs.nvidia.com/nvidia-sitemap-latest.xml",
    "https://docs.nvidia.com/nvidia-sitemap-202310.xml",
    "https://docs.nvidia.com/nvda-sitemap1.xml",
    "https://docs.nvidia.com/nvda-sitemap2.xml",
]

In [ ]:
import nest_asyncio

nest_asyncio.apply()

from langchain.document_loaders.sitemap import SitemapLoader


def prepare_sitemap_loader(link):
    sitemap_loader = SitemapLoader(
        link,
        # filter_urls=["https://api.python.langchain.com/en/latest/"],
        parsing_function=remove_nav_and_header_elements,
        continue_on_failure=True,
        is_local=True,
    )
    sitemap_loader.requests_per_second = 4
    return sitemap_loader


In [ ]:
import time

# docset = []
# docset.extend(docs)
local_file = "NVIDIA.xml"
# for sitemap_url in sitemap_urls:
sitemap_loader = prepare_sitemap_loader(local_file)
docset = sitemap_loader.load()
# docset.extend(docset)
# time.sleep(120)
# process_sitemap(sitemap_url)

print("Sitemaps processed")


In [ ]:
len(docset)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Our sentences we like to encode
sentences = [
    "This framework generates embeddings for each input sentence",
    "Sentences are passed as a list of string.",
    "The quick brown fox jumps over the lazy dog.",
]

# Sentences are encoded by calling model.encode()
embeddings = model.encode(sentences)

# Print the embeddings
for sentence, embedding in zip(sentences, embeddings):
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print("")


In [ ]:
from langchain.vectorstores import Chroma
from langchain.vectorstores import FAISS

# from langchain.document_loaders import WebBaseLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
# split_docset into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_chunks = text_splitter.split_documents(docset)
len(text_chunks)

In [ ]:
vector_db = None
# https://api.python.langchain.com/en/latest/embeddings/langchain.embeddings.huggingface.HuggingFaceEmbeddings.html
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "xla"}
encode_kwargs = {"normalize_embeddings": False}
embedding = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)


In [ ]:
persist_directory = "chroma_db"
vector_db = Chroma.from_documents(documents=text_chunks, embedding=embedding, persist_directory=persist_directory)


In [ ]:
# Persist the db to disk
vector_db.persist()

In [ ]:
%pip install langchainhub
%pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.8/479.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 53.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 80.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 593.7/593.7 kB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.2

In [ ]:
# Retrieve Vector embeddings
if vector_db is not None:
    vector_retriever = vector_db.as_retriever()
else:
    # If variables are cleared,
    vector_db_persist = Chroma(persist_directory="/content/drive/MyDrive/Colab_Notebooks/nlp/chroma_db", embedding_function=embedding)
    vector_retriever = vector_db_persist.as_retriever()


In [ ]:
# Test retriever
vector_retriever.get_relevant_documents("my query")

RuntimeError: ignored

In [ ]:
# RAG prompt
from langchain import hub

# llama_prompt = hub.pull("rlm/rag-prompt-llama")
llama_prompt = "Explain why Tesla is not the scientist TESLA"
llama_prompt_cuda = "What is the NVIDIA CUDA Toolkit?"


In [ ]:
# Build the CPP inference mode LLM
from langchain.chains import RetrievalQA
from langchain.llms import LlamaCpp

# from langchain.callbacks.manager import CallbackManager
# from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# https://python.langchain.com/docs/guides/local_llms#llamacpp
llama = LlamaCpp(
    model_path="/home/ec2-user/llama.cpp/models/ggml-vocab-llama.gguf",
    n_gpu_layers=1,
    n_batch=512,
    n_ctx=2048,
    f16_kv=True,
    verbose=True,
)


In [ ]:
%pip install accelerate
%pip install bitsandbytes

In [ ]:
%load_ext autoreload

In [ ]:
# Use python version per llama docs
import os
from dotenv import load_dotenv
import torch

load_dotenv()
from torch import cuda, bfloat16
from transformers import BitsAndBytesConfig, AutoConfig, AutoModelForCausalLM

model_id = "meta-llama/Llama-2-7b-chat-hf"
# use cpu only
device = f"cuda:{cuda.current_device()}" if cuda.is_available() else "cpu"

# quantization level
quantization_config = BitsAndBytesConfig(load_in_8bit_fp32_cpu_offload=False, llm_int8_threshold=200.0)


# begin initializing HF items, you need an access token
hf_auth = os.getenv("HF_AUTH_TOKEN")
model_config = AutoConfig.from_pretrained(model_id, use_auth_token=hf_auth)

llama = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=bfloat16,
    trust_remote_code=True,
    config=model_config,
    quantization_config=quantization_config,
    device_map="auto",
    use_auth_token=hf_auth,
)
with torch.no_grad():
    llama.eval()
print(f"Model is running on: {device}")

In [ ]:
# Get memory footprint
print(llama.get_memory_footprint())


In [ ]:
# Use tokenizer from pretained model
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=hf_auth)
# Stopping Criteria to reduce rambling
stop_list = ["\nHuman:", "\n```\n"]
stop_token_ids = [tokenizer(stop_tkn)["input_ids"] for stop_tkn in stop_list]
stop_token_ids

In [ ]:
# Convert to LongTensor format
from torch import tensor, long
print("device: ",device)
stop_token_ids = [tensor(id).to(device=device, dtype=long) for id in stop_token_ids]
stop_token_ids


In [ ]:
import torch
from transformers import StoppingCriteria, StoppingCriteriaList


# Overwrite EOS class
class EOS(StoppingCriteria):
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # return super().__call__(input_ids, scores, **kwargs)
        # input_ids = input_ids[0].to(device=device)
        for stop_id in stop_token_ids:
            if torch.eq(input_ids[0][-len(stop_id) :], stop_id).all():
                return True
        return False


stopping_criteria = StoppingCriteriaList([EOS()])

In [ ]:
from transformers import pipeline

text_generator = pipeline(
    model=llama,
    tokenizer=tokenizer,
    return_full_text=True,
    task="text-generation",
    stopping_criteria=stopping_criteria,
    temperature=0.3,
    max_new_tokens=512,
    repetition_penalty=1.1,
)


In [ ]:
test_result = text_generator(llama_prompt)
print(test_result[0]["generated_text"])
del test_result


In [ ]:
# Use a pipeline as a high-level helper
# from transformers import pipeline
# pipe = pipeline("text-generation", model=model_id)
from langchain.llms import HuggingFacePipeline

llm_llama2 = HuggingFacePipeline(pipeline=text_generator)


In [ ]:
# checking again that everything is working fine
llm_llama2(prompt=llama_prompt_cuda)

In [ ]:
from langchain.chains import ConversationalRetrievalChain

conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm_llama2, retriever=vector_retriever, return_source_documents=True
)


In [ ]:
# RetrievalQA
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_llm(llm=llm_llama2, retriever=vector_retriever, return_source_documents=True)



In [ ]:
dir()[:15]

In [ ]:
%load_ext autoreload

In [ ]:
import gc
import torch
def report_gpu():
   print(torch.cuda.list_gpu_processes())
   gc.collect()
   torch.cuda.empty_cache()

report_gpu()

In [ ]:
history=[]
question="Tell me about CUDA for Windows."
result = conv_chain({'question':question,'chat_history':history})
print(result["answer"])

In [ ]:
history=[(question,result['answer'])]
question="What is the NVIDIA CUDA Toolkit?\n"
result = conv_chain({'question':question,'chat_history':history})
print(result["answer"])

In [ ]:
print(result["source_documents"])


In [ ]:
len(multi_html_docs)
multi_html_docs[0]

In [ ]:
# sample = "https://phet-dev.colorado.edu/html/build-an-atom/0.0.0-3/simple-text-only-test-page.html"
sample = "https://www.educative.io/answers/how-to-write-hello-world-in-html"
process_html(sample)

In [ ]:
feature_extractor = MarkupLMFeatureExtractor()
feature_ext_encoding = feature_extractor(multi_html_docs)

In [ ]:
# Now process the entire corpus with MarkupLMFeatureExtractor
# encoding = feature_extractor(multi_html_strings)
processor = MarkupLMProcessor.from_pretrained("microsoft/markuplm-base")

In [ ]:
auto_processor = AutoProcessor.from_pretrained("microsoft/markuplm-base-finetuned-websrc")


In [ ]:
from torch.utils.data import Dataset


class MarkupLMDataset(Dataset):
    """Dataset for token classification with MarkupLM."""

    def __init__(self, data, processor=None, max_length=512):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # first, get nodes, xpaths and node labels
        item = self.data[idx]
        nodes, xpaths, node_labels = item["nodes"], item["xpaths"], item["node_labels"]

        # provide to processor
        encoding = self.processor(
            nodes=nodes,
            xpaths=xpaths,
            node_labels=node_labels,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # remove batch dimension
        encoding = {k: v.squeeze() for k, v in encoding.items()}

        return encoding

In [ ]:
dataset = MarkupLMDataset(data=multi_html_docs, processor=auto_processor, max_length=512)


In [ ]:
question = "is there a page?"
encoding = auto_processor(multi_html_docs[0], questions=question, return_tensors="pt")
print(encoding.keys())  # Outputs: dict_keys(['nodes', 'xpaths'])

In [ ]:
encoding.input_ids[0]

In [ ]:
encoding.attention_mask[0]

In [ ]:
import torch

model = MarkupLMForQuestionAnswering.from_pretrained("microsoft/markuplm-base-finetuned-websrc")

with torch.no_grad():
    outputs = model(**encoding)

# print("Start Logits:", outputs.start_logits)
# print("End Logits:", outputs.end_logits)

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = encoding.input_ids[0, answer_start_index : answer_end_index + 1]
auto_processor.decode(predict_answer_tokens).strip()


In [ ]:
node_labels = [[]]
for node_text in encoding["nodes"][0]:
    print(node_text)

In [ ]:
feature_extractor = MarkupLMFeatureExtractor()

# List sites for encoding
site_1 = "docs.nvidia.com"
site_2 = "forums.developer.nvidia.com"
site_lists = [site_1, site_2]

# Extract features from site_list, BeautifulSoup is under the hood.
multi_html_strings = []

for site in site_lists:
    with open(site) as f:
        multi_html_strings.append(f.read())

encoding = feature_extractor(multi_html_strings)

In [ ]:
tokenizer = MarkupLMTokenizerFast.from_pretrained("microsoft/markuplm-base")
processor = MarkupLMProcessor(encoding, tokenizer)

In [ ]:
# from transformers import MarkupLMProcessor, MarkupLMForQuestionAnswering

processor = MarkupLMProcessor.from_pretrained("microsoft/markuplm-base")
model = MarkupLMForQuestionAnswering.from_pretrained("microsoft/markuplm-base-finetuned-websrc")

html_string = """
 <!DOCTYPE html>
 <html>
 <head>
 <title>Hello world</title>
 </head>
 <body>
 <h1>Welcome</h1>
 <p>My name is Niels.</p>
 </body>
 </html>"""

question = "What's his name?"
x_encoding = processor(html_string, questions=question, return_tensors="pt")
print(x_encoding.keys())


In [ ]:
x_encoding.inputs_ids[0]

In [ ]:
import torch

model = MarkupLMForQuestionAnswering.from_pretrained("microsoft/markuplm-base-finetuned-websrc")

with torch.no_grad():
    outputs = model(**x_encoding)

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = x_encoding.input_ids[0, answer_start_index : answer_end_index + 1]
processor.decode(predict_answer_tokens).strip()


In [ ]:
from transformers import AutoProcessor, MarkupLMForQuestionAnswering

# pre-processor to handle text and images
# https://huggingface.co/docs/transformers/main/en/autoclass_tutorial#autoprocessor
processor = AutoProcessor.from_pretrained("microsoft/markuplm-base-finetuned-websrc")
model = MarkupLMForQuestionAnswering.from_pretrained("microsoft/markuplm-base-finetuned-websrc")

html_string = """
<!DOCTYPE html>
 <html>
 <head>
 <title>Hello world</title>
 </head>
 <body>
 <h1>Welcome</h1>
 <p>My name is Niels.</p>
 </body>
 </html>,
 <!DOCTYPE html>
 <html>
 <head>
 <title>Hello world again</title>
 </head>
 <body>
 <h1>Welcome</h1>
 <p>My dog is Wralph.</p>
 </body>
 </html>"""

question = "What's his dog?"
auto_encoding = processor(html_string, questions=question, return_tensors="pt")
print(auto_encoding.keys())


In [ ]:
import torch

with torch.no_grad():
    outputs = model(**auto_encoding)

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = auto_encoding.input_ids[0, answer_start_index : answer_end_index + 1]
processor.decode(predict_answer_tokens).strip()
